In [87]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [88]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:
queryEmployee = """
SELECT 
[BusinessEntityID]
      ,[NationalIDNumber]
      ,[LoginID]
      ,[BirthDate]
      ,[MaritalStatus]
      ,[Gender]
      ,[HireDate]
      ,[SalariedFlag]
      ,[VacationHours]
      ,[SickLeaveHours]
      ,[CurrentFlag]
FROM HumanResources.Employee
"""

tablaEmployee = pd.read_sql_query(queryEmployee, motorBaseDatos)



queryEmployeePayHistory = """
SELECT 
[BusinessEntityID]
      ,[Rate]
      ,[PayFrequency]
FROM HumanResources.EmployeePayHistory
"""
tablaEmployeePayHistory = pd.read_sql_query(queryEmployeePayHistory, motorBaseDatos)



queryEmployeeDepartmentHistory = """
SELECT 
[BusinessEntityID]
      ,[DepartmentID]
      ,[StartDate]
      ,[EndDate]
FROM HumanResources.EmployeeDepartmentHistory
"""
tablaEmployeeDepartmentHistory = pd.read_sql_query(queryEmployeeDepartmentHistory, motorBaseDatos)

queryDepartment = """
SELECT 
[DepartmentID]
      ,[Name]
FROM HumanResources.Department
"""
tablaDeparment = pd.read_sql_query(queryDepartment, motorBaseDatos)




queryPerson = """
SELECT 
      [BusinessEntityID],
      [NameStyle],
      [Title],
      [FirstName],
      [MiddleName],
      [LastName]
FROM Person.Person
"""
tablaPerson = pd.read_sql_query(queryPerson, motorBaseDatos)


queryPersonPhone = """
SELECT 
[BusinessEntityID]
      ,[PhoneNumber]
FROM Person.PersonPhone
"""
tablaPersonPhone = pd.read_sql_query(queryPersonPhone, motorBaseDatos)


queryEmailAddress = """
SELECT 
[BusinessEntityID]
      ,[EmailAddress]
FROM Person.EmailAddress
"""
tablaEmailAddress = pd.read_sql_query(queryEmailAddress, motorBaseDatos)



queryAddress = """
SELECT 
[AddressID]
      ,[StateProvinceID]
FROM Person.Address
"""
tablaAddress = pd.read_sql_query(queryAddress, motorBaseDatos)


queryBusinessEntityAddress = """
SELECT 
[BusinessEntityID]
      ,[AddressID]
FROM Person.BusinessEntityAddress
"""
tablaBusinessEntityAddress = pd.read_sql_query(queryBusinessEntityAddress, motorBaseDatos)



queryStateProvince = """
SELECT 
[StateProvinceID]
      ,[TerritoryID]
FROM Person.StateProvince
"""
tablaStateProvince = pd.read_sql_query(queryStateProvince, motorBaseDatos)


# tablaEmployee
# tablaPerson
# tablaEmployeePayHistory
# tablaEmployeeDepartmentHistory
# tablaDeparment
# tablaPersonPhone
# tablaEmailAddress
# tablaAddress
# tablaBusinessEntityAddress
# tablaStateProvince

MERGE PARA CONSOLIDAR TABLAS

In [90]:
# PARA OBTENER EL NOMBRE DEL DEPARTAMENTO DE LA PERSONAS

# SE UNE LA TABLA EmployeeDepartmentHistory CON LA TABLA Department PARA UNIR StartDate,EndDate,Name(Nombre del departamento)
department = tablaEmployeeDepartmentHistory.merge(tablaDeparment, on='DepartmentID')


department.rename(columns={
    'Name' :'DepartmentName'
}, inplace=True)


# LA COLUMNA DepartmentID  YA NO ES NECESARIA
department.drop(columns=[
    'DepartmentID'
], inplace=True)


department

,BusinessEntityID,StartDate,EndDate,DepartmentName
0,1,2009-01-14,None,Executive
1,2,2008-01-31,None,Engineering
2,3,2007-11-11,None,Engineering
3,4,2007-12-05,2010-05-30,Engineering
4,4,2010-05-31,None,Tool Design
...,...,...,...,...
291,286,2013-05-30,None,Sales
292,287,2012-04-16,None,Sales
293,288,2013-05-30,None,Sales
294,289,2012-05-30,None,Sales


In [91]:
# PARA OBTENER EL TERRITORY-ID DE LA PERSONAS

# SE UNE LA TABLA BusinessEntityAddress CON LA TABLA Address PARA RELACIONAR EL AddresID CON LA PERSONA
address = tablaBusinessEntityAddress.merge(tablaAddress, on='AddressID')

# LA COLUMNA AddressID  YA NO ES NECESARIA
address.drop(columns=[
    'AddressID'
], inplace=True)



# SE UNE LA NUEVA TABLA address CON COLUMNAS (BusinessEntityID,StateProvinceID) CON LA TABLA StateProvince PARA RELACIONAR EL TerritoryID CON LA PERSONA
address = address.merge(tablaStateProvince, on='StateProvinceID')


address.rename(columns={
    'TerritoryID' : 'SalesTerritoryKey'
}, inplace=True)


# LA COLUMNA StateProvinceID  YA NO ES NECESARIA
address.drop(columns=[
    'StateProvinceID'
], inplace=True)

address

,BusinessEntityID,SalesTerritoryKey
0,1,1
1,2,1
2,3,1
3,4,3
4,5,1
...,...,...
19609,20099,1
19610,20305,1
19611,20419,1
19612,20550,1


TRANSFORMACION

UNION TABLAS

In [92]:
dimensionEmployee = tablaEmployee.merge(tablaPerson, on='BusinessEntityID')
dimensionEmployee = dimensionEmployee.merge(tablaPersonPhone, on='BusinessEntityID')
dimensionEmployee = dimensionEmployee.merge(tablaEmailAddress, on='BusinessEntityID')
dimensionEmployee = dimensionEmployee.merge(tablaEmployeePayHistory, on='BusinessEntityID')
dimensionEmployee = dimensionEmployee.merge(department, on='BusinessEntityID')
dimensionEmployee = dimensionEmployee.merge(address, on='BusinessEntityID')
dimensionEmployee

,BusinessEntityID,NationalIDNumber,LoginID,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,...,MiddleName,LastName,PhoneNumber,EmailAddress,Rate,PayFrequency,StartDate,EndDate,DepartmentName,SalesTerritoryKey
0,1,295847284,adventure-works\ken0,1969-01-29,S,M,2009-01-14,True,99,69,...,J,Sánchez,697-555-0142,ken0@adventure-works.com,125.5000,2,2009-01-14,None,Executive,1
1,2,245797967,adventure-works\terri0,1971-08-01,S,F,2008-01-31,True,1,20,...,Lee,Duffy,819-555-0175,terri0@adventure-works.com,63.4615,2,2008-01-31,None,Engineering,1
2,3,509647174,adventure-works\roberto0,1974-11-12,M,M,2007-11-11,True,2,21,...,None,Tamburello,212-555-0187,roberto0@adventure-works.com,43.2692,2,2007-11-11,None,Engineering,1
3,4,112457891,adventure-works\rob0,1974-12-23,S,M,2007-12-05,False,48,80,...,None,Walters,612-555-0100,rob0@adventure-works.com,8.6200,2,2007-12-05,2010-05-30,Engineering,3
4,4,112457891,adventure-works\rob0,1974-12-23,S,M,2007-12-05,False,48,80,...,None,Walters,612-555-0100,rob0@adventure-works.com,8.6200,2,2010-05-31,None,Tool Design,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,286,758596752,adventure-works\lynn0,1977-02-14,S,F,2013-05-30,True,36,38,...,N,Tsoflias,1 (11) 500 555-0190,lynn0@adventure-works.com,23.0769,2,2013-05-30,None,Sales,9
330,287,982310417,adventure-works\amy0,1957-09-20,M,F,2012-04-16,True,21,30,...,E,Alberts,775-555-0164,amy0@adventure-works.com,48.1010,2,2012-04-16,None,Sales,1
331,288,954276278,adventure-works\rachel0,1975-07-09,S,F,2013-05-30,True,35,37,...,B,Valdez,1 (11) 500 555-0140,rachel0@adventure-works.com,23.0769,2,2013-05-30,None,Sales,8
332,289,668991357,adventure-works\jae0,1968-03-17,M,F,2012-05-30,True,37,38,...,B,Pak,1 (11) 500 555-0145,jae0@adventure-works.com,23.0769,2,2012-05-30,None,Sales,10


CAMBIO DE NOMBRE

In [93]:
dimensionEmployee.rename(columns={
    'BusinessEntityID' : 'EmployeeKey',
    'NationalIDNumber' : 'EmployeeNationalIDAlternateKey',
    'PhoneNumber' :'Phone',
    'Rate' : 'BaseRate'
}, inplace=True)



dimensionEmployee["ParentEmployeeKey"] = None
dimensionEmployee["ParentEmployeeNationalIDAlternateKey"] = None
dimensionEmployee["EmergencyContactName"] = dimensionEmployee["FirstName"] + " "+ dimensionEmployee["LastName"]
dimensionEmployee["EmergencyContactPhone"] = dimensionEmployee["Phone"]
dimensionEmployee["EmployeePhoto"] = None
dimensionEmployee["SalesPersonFlag"] = None

dimensionEmployee.loc[dimensionEmployee["EndDate"].isnull(), "Status"] = "Current"
dimensionEmployee.loc[dimensionEmployee["EndDate"].notnull(), "Status"] = None


# print(dimensionEmployee.columns)
dimensionEmployee

,EmployeeKey,EmployeeNationalIDAlternateKey,LoginID,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,...,EndDate,DepartmentName,SalesTerritoryKey,ParentEmployeeKey,ParentEmployeeNationalIDAlternateKey,EmergencyContactName,EmergencyContactPhone,EmployeePhoto,SalesPersonFlag,Status
0,1,295847284,adventure-works\ken0,1969-01-29,S,M,2009-01-14,True,99,69,...,None,Executive,1,None,None,Ken Sánchez,697-555-0142,None,None,Current
1,2,245797967,adventure-works\terri0,1971-08-01,S,F,2008-01-31,True,1,20,...,None,Engineering,1,None,None,Terri Duffy,819-555-0175,None,None,Current
2,3,509647174,adventure-works\roberto0,1974-11-12,M,M,2007-11-11,True,2,21,...,None,Engineering,1,None,None,Roberto Tamburello,212-555-0187,None,None,Current
3,4,112457891,adventure-works\rob0,1974-12-23,S,M,2007-12-05,False,48,80,...,2010-05-30,Engineering,3,None,None,Rob Walters,612-555-0100,None,None,None
4,4,112457891,adventure-works\rob0,1974-12-23,S,M,2007-12-05,False,48,80,...,None,Tool Design,3,None,None,Rob Walters,612-555-0100,None,None,Current
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,286,758596752,adventure-works\lynn0,1977-02-14,S,F,2013-05-30,True,36,38,...,None,Sales,9,None,None,Lynn Tsoflias,1 (11) 500 555-0190,None,None,Current
330,287,982310417,adventure-works\amy0,1957-09-20,M,F,2012-04-16,True,21,30,...,None,Sales,1,None,None,Amy Alberts,775-555-0164,None,None,Current
331,288,954276278,adventure-works\rachel0,1975-07-09,S,F,2013-05-30,True,35,37,...,None,Sales,8,None,None,Rachel Valdez,1 (11) 500 555-0140,None,None,Current
332,289,668991357,adventure-works\jae0,1968-03-17,M,F,2012-05-30,True,37,38,...,None,Sales,10,None,None,Jae Pak,1 (11) 500 555-0145,None,None,Current


CARGAR A LA BODEGA

In [94]:
dimensionEmployee.to_sql('dimensionEmployee',motorBodegaDatos, if_exists='replace',index=False)

66